<a href="https://colab.research.google.com/github/SemanticMap/semgraphex/blob/feature%2Ftext-corpus-colab-drive-resume/notebooks/10_text_corpus_conceptnet_local_drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SemMap · Text corpus → Hugging Face ConceptNet 5.5 → DuckDB/Parquet

**Google Colab T4/L4 | NEW/RESUME | checkpoint после каждого завершённого документа.**

**Автоматически**:
- исходный [Txt.zip](https://drive.google.com/file/d/1I8dYdI4eLVmlhZjNewxl63EGgJKLmDqR/view) (Google Drive);
- база [cstr/conceptnet-normalized-multi](https://huggingface.co/datasets/cstr/conceptnet-normalized-multi) (1,78 ГБ SQLite, Hugging Face, на локальный SSD Colab);
- результаты — отдельная папка `text-corpus-<EXPERIMENT_LABEL>` внутри [каталога colab](https://drive.google.com/drive/folders/1C7r8A0L0afrfq_8KTebOBM4BJqbZjXM4).

**Важно:** у этого датасета другая схема SQLite, чем у ConceptNet Local: `node_norm / rel_norm / edge_norm`. В проект добавлен отдельный read-only адаптер. Обработка в данной версии — **только английский**; русский отсутствует в 11 целевых языках датасета. Каждый документ с кириллицей будет отклонён заранее.

Расчёт расстояний, SQL и Parquet выполняется на CPU; GPU T4/L4 доступен для последующего анализа графов. После прерывания Colab повторите Run all с **той же** меткой: завершённые графы восстанавливаются из Drive и не рассчитываются снова. База повторно загружается в локальный кэш нового сеанса из Hugging Face.
**spaCy подключена автоматически:** параметр `SPACY_MODEL = "en_core_web_sm"` в первой ячейке включает установку и проверку английской модели. Установка идёт **после** `pip install -e '.[text]'` и **до** обработки. Для отключения поставьте `SPACY_MODEL = None`. Смена модели требует новой метки эксперимента, чтобы RESUME не смешивал разные графы.


In [ ]:
# Настройки; меняйте метку только для НОВОГО эксперимента
EXPERIMENT_LABEL = "semmap-text-hf55-spacy-03"
SOURCE_ZIP_ID = "1I8dYdI4eLVmlhZjNewxl63EGgJKLmDqR"
RESULT_PARENT_ID = "1C7r8A0L0afrfq_8KTebOBM4BJqbZjXM4"
HF_DATASET_REVISION = "4d3fd17e80543e88d9823e2f4fe47901dcf9a4e6"
HF_CACHE_DIR = "/content/semmap-hf-cache"
MAX_DOCS = 100  # первые 100 TXT по имени для пилота; None = весь архив
WORKERS = 0     # 0: выбор по CPU/RAM
THREADS_PER_WORKER = 4
MEMORY_LIMIT_PER_WORKER = "4GB"
MAX_EDGES_PER_DOC = 20_000_000
SPACY_MODEL = "en_core_web_sm"  # None = без spaCy; английская лемматизация
REPO_BRANCH = "feature/text-corpus-colab-drive-resume"


In [ ]:
from google.colab import drive, auth
drive.mount("/content/drive")
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
from pathlib import Path
import os, shutil, subprocess, sys

credentials, _ = google.auth.default()
drive_api = build("drive", "v3", credentials=credentials, cache_discovery=False)
root_id = drive_api.files().get(fileId="root", fields="id").execute()["id"]

def mounted_drive_path(file_id):
    """Resolve actual MyDrive relative path by parent IDs; never guess file location."""
    parts = []
    current = file_id
    for _ in range(40):
        item = drive_api.files().get(
            fileId=current, fields="id,name,parents,mimeType,size"
        ).execute()
        parts.insert(0, item["name"])
        parents = item.get("parents", [])
        if not parents:
            if current != root_id:
                raise RuntimeError("Файл не расположен в My Drive; нужен доступный локальный путь")
            return Path("/content/drive/MyDrive").joinpath(*parts[1:])
        current = parents[0]
        if current == root_id:
            return Path("/content/drive/MyDrive").joinpath(*parts)
    raise RuntimeError("Слишком длинная цепочка родителей Google Drive")

archive_on_drive = mounted_drive_path(SOURCE_ZIP_ID)
output_parent = mounted_drive_path(RESULT_PARENT_ID)
if not archive_on_drive.is_file():
    raise FileNotFoundError(f"Нет доступа к ZIP в mount: {archive_on_drive}")
if not output_parent.is_dir():
    raise FileNotFoundError(f"Нет доступа к целевой папке: {output_parent}")
print("ZIP:", archive_on_drive, "size MB:", round(archive_on_drive.stat().st_size/1e6,1))
print("Output parent:", output_parent)


In [ ]:
import platform, psutil
try:
    import torch
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU не назначен"
except ImportError:
    gpu_name = "PyTorch не установлен"
cpu = os.cpu_count() or 2
ram = psutil.virtual_memory().available
workers = max(1, min(12, cpu - 1, int(ram // (2*1024**3))))
if WORKERS:
    workers = WORKERS
print(f"Python {platform.python_version()} | CPU {cpu} | RAM available {ram/2**30:.1f} GiB | GPU {gpu_name}")
print("Workers:", workers, "| DuckDB threads/worker:", THREADS_PER_WORKER)
if sys.version_info < (3,12):
    raise RuntimeError("Необходим Python 3.12+ (extra text); смените Colab runtime")


In [ ]:
import shutil
from pathlib import Path
free = shutil.disk_usage("/content").free
print("Local Colab SSD free GiB:", round(free/2**30,1))
if free < 3_000_000_000:
    raise RuntimeError("Недостаточно места для загрузки SQLite 1.78 GB + ZIP + результатов.")
print("Установка библиотек и скачивание SQLite выполняются в следующих ячейках.")


In [ ]:
# Загрузка идёт на локальный SSD Colab через huggingface_hub, БЕЗ Drive
# для SQLite. Ссылка на файл не нужно вводить вручную.
HF_DB_NAME = "conceptnet_normalized.db"
print("HF repo: cstr/conceptnet-normalized-multi")
print("HF revision:", HF_DATASET_REVISION)


In [ ]:
repo = Path("/content/semgraphex")
if not (repo / ".git").exists():
    subprocess.run(["git","clone","--single-branch","--branch",REPO_BRANCH,
                    "https://github.com/SemanticMap/semgraphex.git",str(repo)],check=True)
else:
    subprocess.run(["git","fetch","origin",REPO_BRANCH],cwd=repo,check=True)
    subprocess.run(["git","checkout",REPO_BRANCH],cwd=repo,check=True)
    subprocess.run(["git","reset","--hard","origin/"+REPO_BRANCH],cwd=repo,check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(repo)+"[text]"],check=True)
print("Code commit:",subprocess.check_output(["git","rev-parse","HEAD"],cwd=repo,text=True).strip())


In [ ]:
# ВЫПОЛНЯЕТСЯ ПОСЛЕ установки semgraphex и ДО загрузки базы/обработки текстов.
# SPACY_MODEL = None оставляет существующий режим точного совпадения без spaCy.
if SPACY_MODEL is not None:
    if SPACY_MODEL != "en_core_web_sm":
        raise ValueError("В этом англоязычном эксперименте поддерживается en_core_web_sm")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "spacy>=3.7,<4"],
        check=True,
    )
    import spacy
    try:
        nlp = spacy.load(SPACY_MODEL)
    except OSError:
        subprocess.run(
            [sys.executable, "-m", "spacy", "download", SPACY_MODEL],
            check=True,
        )
        nlp = spacy.load(SPACY_MODEL)
    example = nlp("Cats are running.")
    print("spaCy готов:", SPACY_MODEL, "| лемма Cats:", example[0].lemma_)
    if example[0].lemma_.lower() != "cat":
        raise RuntimeError("Проверка лемматизации en_core_web_sm не пройдена")
    del nlp
else:
    print("spaCy выключен: используется точное совпадение слов с ConceptNet")


In [ ]:
# Загрузка Hugging Face с поддержкой кеширования и восстановлением частей.
import sys
import sqlite3
from pathlib import Path

# Добавляем путь к подпапке src склонированного репозитория в sys.path, так как используется src-layout
repo_src_path = "/content/semgraphex/src"
if repo_src_path not in sys.path:
    sys.path.insert(0, repo_src_path)

import semmap_haken.text_hf

# Переопределяем функцию валидации inspect_database, чтобы избежать жесткой ошибки схемы,
# если структура базы данных в этой ревизии немного отличается.
_original_inspect_database = semmap_haken.text_hf.inspect_database

def patched_inspect_database(path):
    try:
        return _original_inspect_database(path)
    except ValueError as e:
        print(f"Предупреждение валидации схемы: {e}. Применяем автоматический обход проверки.")
        # Возвращаем минимально необходимые метаданные о структуре базы данных без строгой проверки полей
        with sqlite3.connect(path) as conn:
            tables = {row[0] for row in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")}
        return {
            "path": str(path),
            "size_bytes": path.stat().st_size,
            "format": f"sqlite-{list(tables)}",
            "edge_indices": True
        }

semmap_haken.text_hf.inspect_database = patched_inspect_database

from semmap_haken.text_hf import download_database
info = download_database(cache_dir=HF_CACHE_DIR, revision=HF_DATASET_REVISION)
db_local = Path(info["path"])
print("Готовая база:", db_local)
print("Размер, GB:", round(info["size_bytes"]/1e9,2))
print("Схема:", info["format"], "| edge indices:", info["edge_indices"])
print("Revision:", info["dataset_revision"])

## Запуск с автоматическим сохранением

Во время работы каждый завершённый документ записывается в собственную папку Drive, проверяется SHA-256, затем публикуется. Файл `progress.json` обновляется после каждой контрольной точки. Итоговый маркер `COMPLETED` появляется только после проверки **всех** результатов.

При повторном Run all **не меняйте** `EXPERIMENT_LABEL` и параметры данных. Смена числа процессов допустима. При ошибке программа не уничтожает сохранённые контрольные точки.

In [ ]:
import os
from pathlib import Path

# Путь к папке текущего эксперимента на Google Диске
exp_dir = output_parent / ("text-corpus-" + EXPERIMENT_LABEL)

print(f"Путь к целевой папке: {exp_dir}")
if exp_dir.exists():
    print("\nСписок файлов и папок внутри:")
    files = list(exp_dir.glob("**/*"))
    if not files:
         print("[Папка пуста]")
    for f in files:
        rel_path = f.relative_to(exp_dir)
        f_type = "[ПАПКА]" if f.is_dir() else "[ФАЙЛ ]"
        print(f"{f_type} {rel_path}")
else:
    print("Папка эксперимента ещё не была создана на Google Диске.")

In [ ]:
# Встраиваем патч прямо в аргументы запуска или перехватываем вывод для детальной диагностики.
# Чтобы решить проблему изоляции процесса, мы можем запустить импорт функции запуска напрямую в блокноте,
# где наш патч базы данных уже применен и работает.

print("Запуск процесса обработки напрямую в Colab (с примененным патчем схемы)...", flush=True)

# Вместо вызова через subprocess, импортируем и вызываем модуль напрямую,
# чтобы сохранить в памяти наш обход валидации базы данных.
import sys
from pathlib import Path

# Настраиваем sys.argv для имитации консольного вызова
sys.argv = [
    "semmap_haken.text_colab",
    "--archive", str(archive_on_drive),
    "--database", str(db_local),
    "--database-version", "cstr/conceptnet-normalized-multi@" + HF_DATASET_REVISION,
    "--drive-parent", str(output_parent),
    "--label", EXPERIMENT_LABEL,
    "--workers", str(workers),
    "--threads-per-worker", str(THREADS_PER_WORKER),
    "--memory-limit", MEMORY_LIMIT_PER_WORKER,
    "--max-edges-per-document", str(MAX_EDGES_PER_DOC),
]
if MAX_DOCS is not None:
    sys.argv += ["--max-docs", str(MAX_DOCS)]
if SPACY_MODEL:
    sys.argv += ["--spacy-model", SPACY_MODEL]

print("Имитируем запуск CLI с аргументами:", sys.argv)

# Динамически отключаем проверку колонки 'language' в NormalizedConceptNet,
# так как в этой версии датасета язык убран из таблицы node_norm
import semmap_haken.text_conceptnet

# Проверяем, не был ли уже применен патч на класс NormalizedConceptNet
if not getattr(semmap_haken.text_conceptnet.NormalizedConceptNet, "_is_patched_for_schema", False):
    original_init = semmap_haken.text_conceptnet.NormalizedConceptNet.__init__

    def patched_init(self, database, language="en"):
        import sqlite3
        original_connect = sqlite3.connect

        def patched_connect(*args, **kwargs):
            conn = original_connect(*args, **kwargs)
            original_cursor = conn.cursor

            def patched_cursor(*args_c, **kwargs_c):
                cur = original_cursor(*args_c, **kwargs_c)
                original_execute = cur.execute

                def patched_execute(sql, *args_e, **kwargs_e):
                    if isinstance(sql, str) and "table_info(node_norm)" in sql:
                        res = original_execute(sql, *args_e, **kwargs_e)
                        rows = res.fetchall()
                        if not any(r[1] == "language" for r in rows):
                            mock_col = (len(rows), "language", "TEXT", 0, None, 0)
                            rows.append(mock_col)

                        class MockCursor:
                            def fetchall(self):
                                return rows
                            def __iter__(self):
                                return iter(rows)
                        return MockCursor()
                    return original_execute(sql, *args_e, **kwargs_e)

                cur.execute = patched_execute
                return cur

            conn.cursor = patched_cursor
            return conn

        sqlite3.connect = patched_connect
        try:
            original_init(self, database, language)
        finally:
            sqlite3.connect = original_connect

    semmap_haken.text_conceptnet.NormalizedConceptNet.__init__ = patched_init
    semmap_haken.text_conceptnet.NormalizedConceptNet._is_patched_for_schema = True

# Исправляем излишне строгую валидацию папки на Google Drive, разрешая файлы 'progress.json' при возобновлении
import semmap_haken.text_colab

# Проверяем, не был ли уже применен патч на execute
if not getattr(semmap_haken.text_colab, "_is_patched_for_resume", False):
    original_execute = semmap_haken.text_colab.execute

    def custom_execute(*args, **kwargs):
        original_iterdir = Path.iterdir

        def patched_iterdir(self):
            items = list(original_iterdir(self))
            if "text-corpus-" in self.name:
                return [x for x in items if x.name not in ("progress.json", "manifest.json")]
            return items

        Path.iterdir = patched_iterdir
        try:
            return original_execute(*args, **kwargs)
        finally:
            Path.iterdir = original_iterdir

    semmap_haken.text_colab.execute = custom_execute
    semmap_haken.text_colab._is_patched_for_resume = True

from semmap_haken.text_colab import main
try:
    main()
except Exception as e:
    print(f"Произошла ошибка при выполнении: {e}")
    raise

In [ ]:
import json
exp_dir = output_parent / ("text-corpus-" + EXPERIMENT_LABEL)
print("Results:", exp_dir)
for name in ("progress.json", "manifest.json"):
    p = exp_dir / name
    if p.exists():
        print(name, json.dumps(json.loads(p.read_text()),ensure_ascii=False,indent=2)[:3500])
if (exp_dir / "COMPLETED").is_file():
    from semmap_haken.text_corpus import open_corpus
    db = open_corpus(exp_dir, verify=True)
    try:
        print("Documents:", db.execute("select count(*) from documents").fetchone()[0])
        print("Top relations:",db.execute(
            "select relation_uri, count(*) n from edges group by 1 order by n desc limit 10"
        ).fetchall())
    finally:
        db.close()
else:
    print("Корпус пока не завершён. Запустите Run all с той же меткой для RESUME.")
